# Portfolio Construction Research

**Maintained continuation of the 2024 CQF portfolio-construction study**

This is the repository's authoritative analysis. It develops the original study into a reproducible portfolio research workflow covering Black–Litterman views, minimum-variance and tracking objectives, higher-moment utility, fractional Kelly allocation, beta-neutral betting-against-beta construction, optional Fama–French five-factor regression, chronological walk-forward evaluation, turnover, transaction costs and short-borrow costs.

The original August 2024 notebook is preserved in `research_archive/portfolio_construction_2024.ipynb` as provenance. It is not a second implementation and is not required to run this analysis. Corrections from the original study are declared in `AMENDMENTS.md`.

Run all cells from top to bottom. The default uses the frozen data supplied with the actual submission, without network access. The 49-page report is retained in `research_archive/submission_report_2024.pdf`. For deterministic offline validation, set `DATA_MODE = "synthetic"`.


The missing analyses from the supplied 213-cell notebook have been reconciled below. The final appendix maps all 127 source code cells to maintained equivalents. Core diagnostics precede portfolio comparisons; exploratory context and heuristic experiments follow **Research interpretation and next work**.

## 1. Research configuration

- `DATA_MODE`: `"submission"` for the supplied frozen inputs, `"yahoo"` for a fresh download, or `"synthetic"` for a software check.
- Dates use `YYYY-MM-DD`; the end date is exclusive.
- `TRAINING_DAYS` is the minimum expanding-window history before evaluation.
- Targets are refitted at the first available observation of each calendar month.
- `TRANSACTION_COST_BPS` is charged per unit of one-way turnover.
- `ANNUAL_SHORT_BORROW_BPS` is charged daily against gross short exposure.
- `KELLY_FRACTION` scales a constrained full-Kelly solution; unused wealth stays in cash. It does not force that fraction into risky assets.
- `BAB_GROSS_LIMIT` bounds absolute long-plus-short exposure.
- `FACTOR_FILE` is included with the submission. When supplied, `FACTOR_UNIT` must explicitly state whether its values are percentage points or decimals.


In [ ]:
DATA_MODE = "submission"
START_DATE = "2020-01-01"
END_DATE = "2024-08-23"
ANNUAL_RISK_FREE_RATE = 0.03
MAX_WEIGHT = 0.35
TRAINING_DAYS = 504
TRANSACTION_COST_BPS = 10
ANNUAL_SHORT_BORROW_BPS = 100
KELLY_FRACTION = 0.50
BAB_GROSS_LIMIT = 2.0
FACTOR_FILE = "data/F-F_Research_Data_5_Factors_2x3_monthly.csv"
FACTOR_UNIT = "percent"
RANDOM_SEED = 42

ROLLING_WINDOWS = (30, 60, 90)
RISK_AVERSIONS = (0.5, 1.0, 3.07)
KELLY_COMPARISON_FRACTIONS = (0.50, 0.66, 1.00)
# Optional appendix acquisition; the complete default run stays offline.
RUN_NETWORK_CONTEXT = False
NETWORK_CONTEXT_START = "1996-01-01"


## 2. Load the research modules

`portfolio_tools.py` contains the data, Black–Litterman, optimization, walk-forward and reporting workflow. `research_extensions.py` contains the factor, higher-moment, Kelly and BAB methods. Both modules are covered by behavioral tests.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from portfolio_tools import ASSETS, BENCHMARK, load_prices, analyze, charts, save_run
from research_extensions import factor_regression, load_fama_french_csv, monthly_compound

pd.set_option("display.max_columns", 30)
print("Research modules ready. Data mode:", DATA_MODE)

from submission_data import (verify_submission_files, load_submission_prices,
                             load_inflation, load_liquidity, file_sha256)

import numpy as np
from importlib.metadata import version
from portfolio_tools import aligned_returns, black_litterman
from research_extensions import FACTOR_COLUMNS, fractional_kelly_weights
from research_diagnostics import (
    performance_table, capm_diagnostics, rolling_market_model, select_rolling_window,
    mean_variance_weights, capped_proportional_weights, higher_moment_diagnostics,
    factor_fit_diagnostics, factor_tilt, evaluate_targets, normalize_context,
)
diagnostic_tables, diagnostic_figures = {}, {}
print("Package versions:", {p: version(p) for p in ["numpy", "pandas", "scipy", "yfinance"]})

def show_heatmap(frame, title, key, centered=False):
    fig, ax = plt.subplots(figsize=(max(7, len(frame.columns) * .48), max(5, len(frame) * .38)))
    values = frame.to_numpy(dtype=float)
    limit = np.nanmax(np.abs(values))
    image = ax.imshow(values, aspect="auto", cmap="coolwarm" if centered else "viridis",
                      vmin=-limit if centered else None, vmax=limit if centered else None)
    ax.set_xticks(range(len(frame.columns)), frame.columns, rotation=60, ha="right")
    ax.set_yticks(range(len(frame.index)), frame.index)
    ax.set_title(title)
    fig.colorbar(image, ax=ax, shrink=.8)
    fig.tight_layout()
    diagnostic_figures[key] = fig
    plt.show()
    plt.close(fig)

def show_panels(frame, title, key, ylabel):
    columns = 4
    rows = (len(frame.columns) + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=(15, 2.7 * rows), squeeze=False)
    for ax, name in zip(axes.flat, frame):
        ax.plot(frame.index, frame[name], linewidth=.8)
        ax.set_title(name)
        ax.set_ylabel(ylabel)
        ax.tick_params(axis="x", labelrotation=30)
    for ax in list(axes.flat)[len(frame.columns):]:
        ax.set_visible(False)
    fig.suptitle(title)
    fig.tight_layout()
    diagnostic_figures[key] = fig
    plt.show()
    plt.close(fig)


## 3. Acquire and validate prices

The analysis uses a date-by-symbol adjusted-price table, so returns are calculated within each asset and never across symbol boundaries. Yahoo failures stop the run instead of substituting synthetic observations. The source mode, requested dates, coverage and price-table SHA-256 hash are recorded with every output.

Submission mode reads `Adj Close` from the original 24,486-row ETF cache. It validates duplicate date/symbol keys and prices before pivoting. It never uses the repeated AUM snapshot as a historical series. The actual file ends on 20 August 2024; requesting a later end does not fabricate further observations.


In [ ]:
if DATA_MODE == "submission":
    manifest = verify_submission_files()
prices, provenance = load_prices(DATA_MODE, START_DATE, END_DATE, RANDOM_SEED)
if DATA_MODE == "submission":
    provenance["submission_input_hashes"] = {f["path"]: f["sha256"] for f in manifest["files"]}
print(provenance["label"])
print(f"{len(prices):,} price rows, {prices.shape[1]} asset labels")
print("First date:", provenance["first_price"], "| Last date:", provenance["last_price"])
display(prices.head().round(2))


## Universe, data quality and historical diagnostics

Submission mode retains all 21 ETFs; the allocation universe remains the eight
retrospectively selected assets. Other modes diagnose the nine loaded symbols.
Internal missing prices now stop the analysis: dropping return dates can erase
real gains and losses. Leading/trailing unavailable history is trimmed to common
coverage. Entire missing exchange sessions cannot be detected without a calendar.

Initial-training statistics, CAPM fits and correlations use only the first
`TRAINING_DAYS` observations. Full-history growth and drawdown charts are descriptive
and do not choose the eight assets. The outlier table flags absolute daily moves
over 10% for inspection, without deleting or winsorizing them. Sortino uses the
root mean squared shortfall below the period cash rate over **all** observations.

In [ ]:
universe_prices = load_submission_prices(start=START_DATE, end=END_DATE) if DATA_MODE == "submission" else prices.copy()
universe_daily = aligned_returns(universe_prices)
universe_summary = pd.DataFrame({
    "First price": universe_prices.apply(lambda s: s.first_valid_index().date()),
    "Last price": universe_prices.apply(lambda s: s.last_valid_index().date()),
    "Observations": universe_prices.count(),
    "Missing prices": universe_prices.isna().sum(),
    "Role": ["Allocation" if s in ASSETS else "Benchmark" if s == BENCHMARK else "Wider study universe" for s in universe_prices],
})
initial_universe = universe_daily.iloc[:TRAINING_DAYS]
initial_performance = performance_table(initial_universe, ANNUAL_RISK_FREE_RATE, benchmark=initial_universe[BENCHMARK])
universe_correlation = initial_universe.corr()
outliers = universe_daily.where(universe_daily.abs() > .10).stack().rename("Daily simple return").to_frame()
outliers.index.names = ["Date", "ETF"]
display(universe_summary)
print("Initial training through:", initial_universe.index[-1].date())
display(initial_performance.sort_values("Sharpe", ascending=False).round(4))
print("Absolute daily moves above 10%:", len(outliers))
display(outliers.head(25))
diagnostic_tables.update(universe_coverage=universe_summary, initial_asset_performance=initial_performance,
                         universe_training_correlation=universe_correlation, flagged_daily_moves=outliers,
                         universe_training_covariance=initial_universe.cov() * 252)
show_heatmap(universe_correlation, "Initial-training correlation (not covariance)", "universe_correlation", centered=True)
growth = (1 + universe_daily).cumprod()
drawdowns = growth / growth.cummax().clip(lower=1) - 1
show_panels(growth, "Descriptive ETF wealth from 1; full loaded history", "universe_growth", "Wealth")
show_panels(drawdowns, "Descriptive ETF drawdown from initial capital / prior peak", "universe_drawdowns", "Drawdown")

## Rolling market exposure and CAPM

For each 30/60/90-observation candidate, coefficients at date *t* use observations
strictly before *t*. Window selection compares **return residuals** on identical
initial-training dates; it does not compare beta coefficients with market returns.
The conditional fitted return uses that day's realized SPY return, so this is a
diagnostic fit score, not an implementable return forecast. The selected window is
then fixed for subsequent rolling diagnostics. Daily alpha is multiplied by 252.

SPY and ACWI CAPM examples each estimate beta against their own market proxy.
Current Yahoo beta metadata is confined to the optional appendix.

In [ ]:
candidate_names = [name for name in universe_daily if name != BENCHMARK]
best_window, window_scores = select_rolling_window(
    initial_universe[candidate_names], initial_universe[BENCHMARK], ROLLING_WINDOWS, ANNUAL_RISK_FREE_RATE)
print("Selected on initial training only:", best_window)
display(window_scores)
rolling = rolling_market_model(universe_daily[candidate_names], universe_daily[BENCHMARK], best_window, ANNUAL_RISK_FREE_RATE)
capm_spy = capm_diagnostics(initial_universe[candidate_names], initial_universe[BENCHMARK], ANNUAL_RISK_FREE_RATE)
capm_acwi = capm_diagnostics(initial_universe[candidate_names], initial_universe["ACWI"], ANNUAL_RISK_FREE_RATE)
display(capm_spy.round(4))
diagnostic_tables.update(rolling_window_scores=window_scores, rolling_betas=rolling["beta"],
                         rolling_annual_alphas=rolling["annual_alpha"], initial_capm_spy=capm_spy,
                         initial_capm_acwi=capm_acwi)
show_panels(rolling["beta"], "Trailing beta to SPY; coefficients available before each date", "rolling_betas", "Beta")
show_panels(rolling["annual_alpha"], "Trailing annualized excess-return intercept", "rolling_alphas", "Annual alpha")

## 4. Fit and evaluate the portfolio methods

All strategies follow the same chronological contract. After the initial training history, each monthly target uses only observations strictly before its rebalance date. Asset weights drift between rebalances. The simulation records one-way turnover and deducts the stated trading costs; short portfolios also incur the stated borrow cost.

| Portfolio | Maintained objective |
| --- | --- |
| Equal weight | Equal long-only asset targets |
| Minimum variance | Minimum annualized covariance-based variance |
| BL maximum Sharpe | Maximum expected Sharpe using the Black–Litterman posterior |
| Minimum tracking error | Minimum annualized active-return variance versus SPY |
| Higher-moment utility | Explicit annual mean/variance utility with sample skew reward and excess-kurtosis penalty |
| Fractional Kelly | Constrained full log-growth optimum, scaled by the Kelly fraction, plus cash |
| BAB beta-neutral | Low-beta long/high-beta short legs scaled to zero estimated market beta and bounded gross exposure |

Covariance and correlation remain separate. Expected returns and covariance use annual units; daily performance statistics use the documented 252-observation convention.


In [ ]:
result = analyze(
    prices,
    annual_rf=ANNUAL_RISK_FREE_RATE,
    max_weight=MAX_WEIGHT,
    training_days=TRAINING_DAYS,
    transaction_cost_bps=TRANSACTION_COST_BPS,
    annual_short_borrow_bps=ANNUAL_SHORT_BORROW_BPS,
    kelly_fraction=KELLY_FRACTION,
    bab_gross_limit=BAB_GROSS_LIMIT,
)
print("Aligned returns:", len(result["returns"]))
print("Training observations:", result["training_days"])
print("Walk-forward evaluation:", result["evaluation_start"].date(), "to", result["evaluation_end"].date())
display(result["returns"][ASSETS].corr().round(2))


## 5. Black–Litterman views

The three views are retained as explicit August 2024 research scenarios: GLD over MTUM by 15 percentage points annually, XLK over XLE by 8 points, and SPHD over VUG by 6 points. They are assumptions, not current forecasts.

The prior is $\Pi=\delta\Sigma w_0$. View uncertainty is scaled from $P(\tau\Sigma)P^T$ and the stated confidence. The table reports annual excess-return estimates in consistent decimal units.

The report dates these views to 21 August 2024 (section 4.8, pages 33–34), after the cache ends. Applying them to 2022–2024 is explicitly a retrospective scenario. A prospective test requires prices after that date and a universe fixed before evaluation.


In [ ]:
display(result["views"].style.format({"Annual difference": "{:.1%}", "Assumed confidence": "{:.0%}"}))
display(pd.DataFrame({"Prior excess return": result["prior"],
                      "BL posterior excess return": result["posterior"]}).style.format("{:.2%}"))

### Black–Litterman matrices and return-vector comparison

These tables restore the original explicit P, Q, Omega and posterior-covariance
calculations. All use the **last rebalance's prior window**, whose cutoff is shown.
P maps asset returns into relative views; Q is an annual excess-return difference;
Omega scales view uncertainty from the same covariance. The covariance of the
posterior mean and the predictive covariance `Sigma + mean covariance` are different
objects. The main strategies continue to use estimated return covariance `Sigma`;
the predictive-covariance allocation below is a labelled sensitivity.

Historical means, SPY/ACWI CAPM, equilibrium prior and BL posterior are compared
as annual **excess** returns. Summary averages are explicitly equal weighted.

In [ ]:
last_rebalance = result["weight_history"].index.get_level_values("Date").max()
last_train = result["returns"].loc[result["returns"].index < last_rebalance]
print("Fit cutoff:", last_train.index[-1].date(), "| Target effective:", last_rebalance.date())
bl = black_litterman(result["covariance"], return_details=True)
display(bl["P"])
display(pd.concat([bl["Q"], pd.Series(np.diag(bl["Omega"]), index=bl["Q"].index, name="View uncertainty variance")], axis=1))
display(bl["predictive_covariance"].round(5))
daily_cash = (1 + ANNUAL_RISK_FREE_RATE) ** (1 / 252) - 1
last_spy = capm_diagnostics(last_train[ASSETS], last_train[BENCHMARK], ANNUAL_RISK_FREE_RATE)
last_acwi = capm_diagnostics(last_train[ASSETS], last_train["ACWI"], ANNUAL_RISK_FREE_RATE)
return_comparison = pd.DataFrame({
    "Historical excess": (last_train[ASSETS].mean() - daily_cash) * 252,
    "SPY CAPM excess": last_spy["CAPM annual excess"],
    "ACWI CAPM excess": last_acwi["CAPM annual excess"],
    "Equilibrium prior": bl["prior"], "BL posterior": bl["posterior"],
})
return_comparison["Posterior minus prior"] = bl["posterior"] - bl["prior"]
display(return_comparison.style.format("{:.2%}"))
return_summary = return_comparison.agg(["mean", "std", "min", "max"]).rename(index={"mean": "Equal-weight mean", "std": "Cross-asset standard deviation"})
display(return_summary.style.format("{:.2%}"))
assert np.allclose(bl["posterior"], result["posterior"])
assert np.linalg.eigvalsh(bl["mean_covariance"]).min() >= -1e-10
diagnostic_tables.update(bl_link_matrix=bl["P"], bl_views=bl["Q"].to_frame(), bl_omega=bl["Omega"],
                         bl_mean_covariance=bl["mean_covariance"], bl_predictive_covariance=bl["predictive_covariance"],
                         return_vector_comparison=return_comparison, return_vector_summary=return_summary)

### Allocation sensitivity to expected returns, covariance and risk aversion

The old return-normalization and `weights / risk_aversion` examples are replaced
with constrained mean–variance optimization: maximize `mu'w - lambda/2 w'Sigma w`,
with nonnegative risky weights, the configured cap, and a cash balance. Lambda
values 0.5, 1 and 3.07 retain the original numerical scenarios; they are not
universal investor categories or a substitute for the Kelly objective.

These are latest-fit sensitivity tables, not additional backtested strategies.
The same covariance and constraints isolate the effect of each return vector.

In [ ]:
return_vector_weights = pd.DataFrame({
    label: mean_variance_weights(return_comparison[label], result["covariance"], 3.07, MAX_WEIGHT)
    for label in ["Historical excess", "Equilibrium prior", "BL posterior"]
})
return_vector_weights["Equal prior reference"] = 1 / len(ASSETS)
return_vector_weights["BL predictive-covariance sensitivity"] = mean_variance_weights(
    bl["posterior"], bl["predictive_covariance"], 3.07, MAX_WEIGHT)
return_vector_weights.loc["Cash"] = 1 - return_vector_weights.sum()
aversion_weights = pd.DataFrame({f"lambda={level:g}": mean_variance_weights(
    bl["posterior"], result["covariance"], level, MAX_WEIGHT) for level in RISK_AVERSIONS})
aversion_weights.loc["Cash"] = 1 - aversion_weights.sum()
aversion_metrics = []
for label in aversion_weights:
    w = aversion_weights.loc[ASSETS, label].to_numpy()
    aversion_metrics.append({"Scenario": label, "Expected annual excess": w @ bl["posterior"],
                             "Annual volatility": np.sqrt(w @ result["covariance"].to_numpy() @ w),
                             "Cash": aversion_weights.loc["Cash", label]})
aversion_metrics = pd.DataFrame(aversion_metrics).set_index("Scenario")
display(return_vector_weights.style.format("{:.1%}"))
display(aversion_weights.style.format("{:.1%}"))
display(aversion_metrics.style.format("{:.2%}"))
diagnostic_tables.update(return_vector_allocations=return_vector_weights,
                         risk_aversion_weights=aversion_weights, risk_aversion_metrics=aversion_metrics)
show_heatmap(aversion_weights, "Latest-fit allocations by risk-aversion coefficient", "risk_aversion_weights")

## 6. Latest portfolio targets

The table includes risky assets and cash. Long-only methods sum to 100%; fractional Kelly deliberately retains cash. BAB may contain negative asset weights and an offsetting cash position because it is a beta-neutral long/short construction. The full target history is exported for audit.


In [ ]:
print(provenance["label"])
display(result["weights"].style.format("{:.1%}"))
print("Budget sums:", result["weights"].sum().round(8).to_dict())
print("Latest BAB diagnostics:")
display(result["bab_diagnostics"].style.format({"Beta": "{:.3f}", "Weight": "{:.2%}"}))
print("BAB estimated beta:", round(result["bab_diagnostics"].attrs["portfolio_beta"], 10))
print("BAB gross exposure:", round(result["bab_diagnostics"].attrs["gross_exposure"], 4))


### Co-skewness, co-kurtosis and portfolio-moment verification

The original third/fourth-order tensors are restored using **raw daily central
moments**. Contracting them with the higher-moment optimizer's own weights and then
standardizing must reproduce skewness and excess kurtosis of that portfolio's
sample return series. Population moments are used consistently for this identity;
the optimizer's finite-sample corrected statistics have a different convention.

The displayed pairwise slices are `E[z_i^2 z_j]` and `E[z_i^2 z_j^2]`, where z is an
asset's standardized daily return. The latter is raw co-kurtosis; subtracting 3
from every off-diagonal entry would not define a valid general excess co-moment.

In [ ]:
moment_diagnostics = higher_moment_diagnostics(last_train[ASSETS], result["weights"].loc[ASSETS, "Higher-moment utility"])
display(moment_diagnostics["checks"])
assert moment_diagnostics["checks"]["Absolute difference"].max() < 1e-8
print("Tensor shapes:", moment_diagnostics["third"].shape, moment_diagnostics["fourth"].shape)
diagnostic_tables.update(higher_moment_checks=moment_diagnostics["checks"],
                         pair_coskewness=moment_diagnostics["pair_coskewness"],
                         pair_cokurtosis=moment_diagnostics["pair_cokurtosis"])
show_heatmap(moment_diagnostics["pair_coskewness"], "Pairwise co-skewness E[z_i squared z_j]", "pair_coskewness", centered=True)
show_heatmap(moment_diagnostics["pair_cokurtosis"], "Raw pairwise co-kurtosis E[z_i squared z_j squared]", "pair_cokurtosis")

## 7. Model, implementation and walk-forward results

The model table describes the final pre-trade estimates. The implementation table aggregates one-way turnover and modeled costs. Historical statistics use only later walk-forward returns and include transaction and short-borrow costs for portfolio strategies; SPY is an uncosted reference.

Only numerical parameter fitting is chronological. The universe and views were chosen in August 2024 and applied retrospectively, so these results are not a point-in-time out-of-sample strategy test. The views are fixed scenarios, parameters were not selected through nested validation, and the cost model does not include market impact, taxes or locate constraints.


In [ ]:
print("LATEST MODEL ESTIMATES —", provenance["label"])
display(result["model"].style.format({"Expected annual return": "{:.2%}",
                                      "Annual volatility": "{:.2%}", "Expected Sharpe": "{:.2f}"}))
print("IMPLEMENTATION —", TRANSACTION_COST_BPS, "bps transaction cost;",
      ANNUAL_SHORT_BORROW_BPS, "bps annual short borrow")
display(result["trading"].style.format({"One-way turnover": "{:.2f}",
                                        "Estimated cost": "{:.2%}",
                                        "Estimated short-borrow cost": "{:.2%}",
                                        "Estimated total implementation cost": "{:.2%}",
                                        "Annualized one-way turnover": "{:.2f}",
                                        "Cost assumption (bps)": "{:.0f}"}))
print("NET WALK-FORWARD HISTORY —", provenance["label"])
display(result["historical"].style.format({"Annual mean": "{:.2%}", "CAGR": "{:.2%}",
                                           "Annual volatility": "{:.2%}", "Sharpe": "{:.2f}",
                                           "Max drawdown": "{:.2%}"}))


### Realized BAB exposure and active performance

Zero estimated beta at rebalance does not guarantee zero realized rolling beta.
The net strategy's rolling alpha/beta uses prior observations of the resulting
return stream and the window selected in initial training. Tracking error is the
annualized standard deviation of daily strategy minus SPY returns; information
ratio divides annual active mean by that tracking error. SPY's self-comparison is
undefined for information ratio, so its zero denominator is displayed as missing.

In [ ]:
active_metrics = performance_table(result["portfolio_returns"], ANNUAL_RISK_FREE_RATE,
                                   benchmark=result["portfolio_returns"][BENCHMARK])
bab_rolling = rolling_market_model(result["portfolio_returns"][["BAB beta-neutral"]],
                                    result["portfolio_returns"][BENCHMARK], best_window, ANNUAL_RISK_FREE_RATE)
bab_exposure = pd.concat([bab_rolling["beta"].iloc[:, 0].rename("Beta"),
                          bab_rolling["annual_alpha"].iloc[:, 0].rename("Annual alpha")], axis=1)
display(active_metrics.round(4))
diagnostic_tables.update(active_performance=active_metrics, bab_rolling_exposure=bab_exposure)
show_panels(bab_exposure, "Realized net BAB exposure to SPY", "bab_rolling_exposure", "Coefficient")

### Fractional Kelly: 0.50, 0.66 and 1.00

Each rebalance solves a long-only, unlevered **full-Kelly log-growth problem with
cash available**, using only the prior window. Fractions scale that same full
solution, so 50% Kelly can invest less than 50% of capital when full Kelly retains
cash. The asset cap applies before scaling. This replaces the previous forced
risky-sleeve implementation and the original inverse-covariance/gross-normalization
illustration. Wealth compounds simple returns at the correct timestamp.

Every comparison below includes monthly turnover, trading costs and cash interest
using the main portfolio ledger. It remains subject to the retrospective universe
selection caveat and is not an unconstrained leveraged Kelly solution.

In [ ]:
kelly_dates = result["weight_history"].index.get_level_values("Date").unique()
kelly_targets = {fraction: [] for fraction in KELLY_COMPARISON_FRACTIONS}
for date in kelly_dates:
    train = result["returns"].loc[result["returns"].index < date, ASSETS]
    full = fractional_kelly_weights(train, ANNUAL_RISK_FREE_RATE, fraction=1.0, cap=MAX_WEIGHT)
    for fraction in KELLY_COMPARISON_FRACTIONS:
        target = full * fraction
        kelly_targets[fraction].append({"Date": date, **target.to_dict(), "Cash": 1 - target.sum()})
kelly_returns = pd.DataFrame(index=result["portfolio_returns"].index)
for fraction, records in kelly_targets.items():
    targets = pd.DataFrame(records).set_index("Date")[ASSETS + ["Cash"]]
    series, turnover = evaluate_targets(result["returns"].loc[kelly_returns.index, ASSETS], targets,
                                        ANNUAL_RISK_FREE_RATE, TRANSACTION_COST_BPS, ANNUAL_SHORT_BORROW_BPS)
    label = f"Kelly {fraction:.2f}"
    kelly_returns[label] = series
    diagnostic_tables[f"kelly_{fraction:.2f}_targets"] = targets
    diagnostic_tables[f"kelly_{fraction:.2f}_turnover"] = turnover
if KELLY_FRACTION in KELLY_COMPARISON_FRACTIONS:
    assert np.allclose(kelly_returns[f"Kelly {KELLY_FRACTION:.2f}"], result["portfolio_returns"]["Fractional Kelly"])
kelly_returns[BENCHMARK] = result["portfolio_returns"][BENCHMARK]
kelly_metrics = performance_table(kelly_returns, ANNUAL_RISK_FREE_RATE)
display(kelly_metrics.round(4))
diagnostic_tables.update(kelly_comparison_returns=kelly_returns, kelly_comparison_metrics=kelly_metrics)
fig, ax = plt.subplots(figsize=(11, 5))
(1 + kelly_returns).cumprod().plot(ax=ax)
ax.set(title="Fractional Kelly comparison: net wealth", ylabel="Wealth from 1")
fig.tight_layout()
diagnostic_figures["kelly_comparison"] = fig
plt.show()
plt.close(fig)

## 8. Fama–French five-factor attribution

The supplied factor vintage spans July 1963 through June 2024. Values are converted from percentage points to decimals once. Daily returns are geometrically compounded to months. Both boundary months are conservatively omitted because return-only input does not prove full-month coverage, then only overlapping factor months are fitted.

We report time-series OLS for all 21 ETFs in submission mode (the nine loaded symbols in Yahoo mode) and separately for the seven net strategy return streams plus SPY. This is attribution, not Fama–MacBeth estimation or a factor-based weight adjustment. Alpha is the monthly intercept multiplied by 12. T-statistics use classical IID residual standard errors, without serial-correlation or heteroskedasticity corrections. Sample dates and counts accompany every regression. Synthetic mode skips regression against real factors.


In [ ]:
factor_path = Path(FACTOR_FILE)
factor_results = universe_factor_results = None
if DATA_MODE == "synthetic":
    print("Real-factor attribution omitted for synthetic prices.")
elif factor_path.exists():
    factors = load_fama_french_csv(factor_path, unit=FACTOR_UNIT)
    provenance["factor_file_sha256"] = file_sha256(factor_path)
    provenance["factor_unit"] = FACTOR_UNIT
    monthly_portfolios = monthly_compound(result["portfolio_returns"])
    factor_results = factor_regression(monthly_portfolios, factors)
    print("NET STRATEGIES AND SPY: monthly time-series attribution")
    display(factor_results)
    if universe_prices is not None:
        universe_monthly = monthly_compound(universe_daily)
        universe_factor_results = factor_regression(universe_monthly, factors)
        print(f"LOADED UNIVERSE ({len(universe_prices.columns)} ETFs): monthly time-series attribution")
        display(universe_factor_results)
else:
    raise FileNotFoundError(f"Factor attribution requires the declared file: {factor_path}")


### Factor loadings and fitted-return diagnostics

The original scatter plots are restored as observed versus fitted **excess returns
using all five factors**. A market-only line drawn from a multivariate model would
omit the other four contributions. These are in-sample attribution diagnostics;
their fit is not evidence of forecasting skill. Loadings are dimensionless and
use decimal ETF and factor returns. Synthetic mode does not pair simulated returns
with real factors.

In [ ]:
factor_loadings = None
if DATA_MODE != "synthetic":
    allocation_monthly = monthly_compound(universe_daily[ASSETS])
    asset_factor_results, actual_factor_excess, fitted_factor_excess = factor_fit_diagnostics(allocation_monthly, factors)
    factor_loadings = asset_factor_results[[f"{f} loading" for f in FACTOR_COLUMNS]].copy()
    factor_loadings.columns = FACTOR_COLUMNS
    display(factor_loadings.round(4))
    diagnostic_tables.update(allocation_factor_loadings=factor_loadings,
                             factor_actual_excess=actual_factor_excess, factor_fitted_excess=fitted_factor_excess)
    show_heatmap(factor_loadings, "Five-factor ETF loadings: full overlapping sample", "factor_loadings", centered=True)
    fig, axes = plt.subplots(4, 2, figsize=(11, 13))
    for ax, name in zip(axes.flat, ASSETS):
        x, y = fitted_factor_excess[name], actual_factor_excess[name]
        ax.scatter(x, y, s=12, alpha=.6)
        low, high = min(x.min(), y.min()), max(x.max(), y.max())
        ax.plot([low, high], [low, high], color="red", linewidth=.8)
        ax.set(title=name, xlabel="Fitted excess return (all five factors)", ylabel="Observed excess return")
    fig.suptitle("In-sample factor attribution diagnostics")
    fig.tight_layout()
    diagnostic_figures["factor_fit"] = fig
    plt.show()
    plt.close(fig)
else:
    print("Factor fit plots require actual ETF returns; skipped for synthetic mode.")

## 9. Portfolio charts

The allocation chart includes cash and signed BAB weights. The growth chart covers only the later walk-forward period. Portfolio series are net of the stated modeled costs; SPY is an uncosted reference.


In [ ]:
weight_chart, growth_chart = charts(result, provenance["label"])
plt.show()

## Research interpretation and next work

This implementation makes the portfolio methods and evaluation contract explicit and testable. It does not treat the best historical Sharpe ratio as proof of superiority. Useful next work includes multiple non-overlapping evaluation periods, parameter stability maps, block-bootstrap uncertainty, alternative view-generation processes, point-in-time factor vintages and richer execution-cost models.

### References

- Black, F. and Litterman, R. (1992), *Global Portfolio Optimization*, Financial Analysts Journal, 48(5), 28–43.
- Fama, E. F. and French, K. R. (2015), *A Five-Factor Asset Pricing Model*, Journal of Financial Economics, 116(1), 1–22.
- Frazzini, A. and Pedersen, L. H. (2014), *Betting Against Beta*, Journal of Financial Economics, 111(1), 1–25.
- Kelly, J. L. (1956), *A New Interpretation of Information Rate*, Bell System Technical Journal, 35(4), 917–926.
- `AMENDMENTS.md` documents corrections from the 2024 implementation.

This repository is quantitative research code, not personalized investment advice or evidence of future returns.


## Supplementary experiments and market context

The following source analyses help inspect assumptions but are not inputs to the
seven-strategy allocation process. They are kept executable here, after the research
interpretation, with their data dates and limitations stated. They do not modify
the strategy return streams above.

### A. Submission AUM snapshot and capped allocation illustration

The cache repeats one AUM value per ETF across its history. It does not contain a
dated AUM time series. ETF AUM also is not the market capitalization of a disjoint
set of underlying investments. These snapshot weights therefore remain an
illustration, not historical equilibrium weights. The 20-candidate allocation uses
a 10% cap; the selected eight use `MAX_WEIGHT` because an eight-asset 10% cap cannot
sum to one. Iterative proportional redistribution enforces every cap simultaneously.

In [ ]:
if DATA_MODE == "submission":
    aum_raw = pd.read_csv(Path("data") / "efts_dataset.csv")
    aum_counts = aum_raw.groupby("ETF")["Market Cap"].nunique(dropna=False)
    if not (aum_counts == 1).all():
        raise ValueError("Expected one repeated AUM snapshot per ETF; inspect the source.")
    aum = pd.to_numeric(aum_raw.groupby("ETF")["Market Cap"].first(), errors="raise").drop(index=BENCHMARK)
    if not np.isfinite(aum).all() or (aum <= 0).any():
        raise ValueError("Snapshot AUM must be positive and finite.")
    aum_table = pd.DataFrame({"AUM snapshot (source units)": aum, "Proportional weight": aum / aum.sum(),
                              "Capped candidate weight": capped_proportional_weights(aum, .10)})
    selected_aum = pd.DataFrame({"Proportional snapshot": aum.loc[ASSETS] / aum.loc[ASSETS].sum(),
                                 "Capped snapshot": capped_proportional_weights(aum.loc[ASSETS], MAX_WEIGHT),
                                 "Equal prior used by workflow": 1 / len(ASSETS)})
    display(aum_table)
    display(selected_aum.style.format("{:.1%}"))
    diagnostic_tables.update(aum_snapshot_illustration=aum_table, selected_aum_illustration=selected_aum)
else:
    print("AUM illustration uses only the submission snapshot; omitted in this mode.")

### B. Factor-threshold allocation heuristic

The original rule multiplies an ETF weight by 1.1 for a positive loading and 0.9
for a negative loading exceeding a threshold. A loading's sign alone is not a
forecast of a factor premium. This is a sensitivity experiment, not Fama–MacBeth
estimation or a validated investment rule.

We restore both the original numerical thresholds (+0.010 / -0.008) and an adaptive
absolute-loading mean-plus-standard-deviation threshold. The old thresholds were
chosen using inconsistent units; here they are explicitly **illustrative** on
corrected dimensionless loadings. All five factors contribute; output is capped
and normalized after the multipliers. Fits stop before the last rebalance and
use the available factor vintage. Before/after monthly statistics use 12 periods
per year and the same calibration sample. They are gross in-sample constant-weight
illustrations, not the net walk-forward strategy results.

In [ ]:
if DATA_MODE != "synthetic":
    tilt_monthly = monthly_compound(last_train[ASSETS])
    tilt_fit, tilt_actual, _ = factor_fit_diagnostics(tilt_monthly, factors)
    tilt_loadings = tilt_fit[[f"{f} loading" for f in FACTOR_COLUMNS]].copy()
    tilt_loadings.columns = FACTOR_COLUMNS
    absolute_loadings = tilt_loadings.abs().to_numpy().ravel()
    adaptive_threshold = float(absolute_loadings.mean() + absolute_loadings.std(ddof=0))
    base = pd.Series(1 / len(ASSETS), index=ASSETS)
    tilt_weights = pd.DataFrame({"Equal reference": base,
        "Original numerical thresholds": factor_tilt(base, tilt_loadings, .010, .008, MAX_WEIGHT),
        "Adaptive threshold": factor_tilt(base, tilt_loadings, adaptive_threshold, adaptive_threshold, MAX_WEIGHT)})
    threshold_summary = pd.DataFrame({"Positive threshold": [.010, adaptive_threshold],
                                      "Negative magnitude threshold": [.008, adaptive_threshold]},
                                     index=["Original numerical thresholds", "Adaptive threshold"])
    filtered_loadings = tilt_loadings.where((tilt_loadings >= .010) | (tilt_loadings <= -.008))
    tilt_sample = tilt_monthly.loc[tilt_actual.index]
    tilt_returns = tilt_sample @ tilt_weights
    tilt_metrics = performance_table(tilt_returns, ANNUAL_RISK_FREE_RATE, periods_per_year=12)
    print("Calibration months:", tilt_sample.index.min().date(), "to", tilt_sample.index.max().date())
    display(threshold_summary)
    display(filtered_loadings.round(4))
    display(tilt_weights.style.format("{:.2%}"))
    display(tilt_metrics.round(4))
    diagnostic_tables.update(factor_tilt_thresholds=threshold_summary, factor_tilt_loadings=tilt_loadings,
                             factor_tilt_filtered=filtered_loadings, factor_tilt_weights=tilt_weights,
                             factor_tilt_calibration_metrics=tilt_metrics)
else:
    print("Real-factor weight heuristic omitted for synthetic returns.")

### C. Original quartile low-minus-high-beta factor

This restores the source's top/bottom-quartile equal-weight spread separately from
the main beta-neutral BAB portfolio. Membership is fixed using initial-training
SPY betas, and later daily returns are the equal-weight low-minus-high spread.
This proxy is not beta neutral and excludes rebalancing and borrowing costs.
It is a zero-net-investment factor payoff per unit long/short notional, not a
funded account return. Consequently we plot arithmetic cumulative payoff, not a
compounded wealth curve, and do not subtract the risk-free rate a second time.

In [ ]:
spread_betas = capm_spy.loc[ASSETS, "Beta"]
low_names = spread_betas.index[spread_betas <= spread_betas.quantile(.25)]
high_names = spread_betas.index[spread_betas >= spread_betas.quantile(.75)]
spread_evaluation = result["returns"].loc[result["portfolio_returns"].index, ASSETS]
low_high = pd.DataFrame({"Low-beta leg": spread_evaluation[low_names].mean(axis=1),
                         "High-beta leg": spread_evaluation[high_names].mean(axis=1)})
low_high["Low minus high"] = low_high.iloc[:, 0] - low_high.iloc[:, 1]
spread = low_high["Low minus high"]
spread_summary = pd.DataFrame({"Annual mean payoff": [spread.mean() * 252],
                               "Annual volatility": [spread.std() * np.sqrt(252)],
                               "Zero-cost payoff ratio": [spread.mean() / spread.std() * np.sqrt(252)]})
spread_membership = pd.DataFrame({"Training beta": spread_betas,
                                  "Leg": ["Low" if s in low_names else "High" if s in high_names else "Unused" for s in ASSETS]})
spread_rolling = rolling_market_model(spread.to_frame(), result["portfolio_returns"][BENCHMARK], best_window, annual_rf=0)
display(spread_membership)
display(spread_summary)
diagnostic_tables.update(quartile_spread_membership=spread_membership, quartile_spread_returns=low_high,
                         quartile_spread_summary=spread_summary, quartile_spread_beta=spread_rolling["beta"])
show_panels(pd.concat([spread.cumsum().rename("Cumulative arithmetic payoff"),
                       spread_rolling["beta"].iloc[:, 0].rename("Rolling beta")], axis=1),
             "Uncosted quartile factor: diagnostic only", "quartile_spread", "Value")

### D. Inflation, bank credit and standardized context

The supplied World Bank CPI and BIS bank-credit series retain their own units.
BIS is selected by the full economic metadata key, not row number 142. A second
chart restores the standardized credit illustration. Full-sample normalization
is descriptive and uses future observations relative to earlier chart dates;
it is not a signal. These macro snapshots have no release-date history and are
not used to manufacture historical trading views.

In [ ]:
inflation = liquidity = macro_chart = None
if DATA_MODE == "submission":
    inflation, liquidity = load_inflation(), load_liquidity()
    credit = liquidity.loc["1996Q1":].copy()
    credit.index = credit.index.to_timestamp(how="end")
    standardized_credit = normalize_context(credit)
    macro_chart, axes = plt.subplots(3, 1, figsize=(11, 9))
    inflation.loc[1996:2023].plot(ax=axes[0])
    axes[0].set(title="World Bank global headline inflation", ylabel="Annual inflation (%)")
    credit.plot(ax=axes[1], color="teal")
    axes[1].set(title="BIS global bank credit: selected series", ylabel="Credit (% of GDP)")
    axes[2].bar(standardized_credit.index, standardized_credit, width=60, color="teal")
    axes[2].axhline(0, color="gray", linewidth=.8)
    axes[2].set(title="Same credit series: full-sample normalization", ylabel="Standard deviations")
    macro_chart.tight_layout()
    diagnostic_tables.update(global_inflation=inflation, global_bank_credit=liquidity.to_frame(),
                             standardized_bank_credit=standardized_credit.to_frame())
    diagnostic_figures["macro_context"] = macro_chart
    plt.show()
    plt.close(macro_chart)
else:
    print("Original macro snapshots shown only in submission mode.")

### E. Optional market-price proxies and current provider metadata

Set `RUN_NETWORK_CONTEXT=True` to acquire the source's six equity indices, GLD,
10-year Treasury yield and VIX. These inputs were not supplied as frozen files;
the default therefore skips this section's network calls explicitly. Errors stop
acquisition rather than inventing history. Downloads are a **current provider
vintage** and will be saved with a UTC acquisition time and hash.

An equal-weight average of standardized equity-index levels measures relative
market-price context, not economic growth. Gold is not an inflation series;
Treasury yield is not a liquidity quantity. Chart labels reflect those distinctions.
Equity indices are aligned on common observed dates for this display only.
The normalization uses the full downloaded sample and does not enter the backtest.

The separate metadata table restores provider beta/AUM inspection with Yahoo's
`beta3Year` and `totalAssets` fields, plus a recent ^IRX quoted annual discount yield.
Missing fields remain missing. The source's hard-coded Kavout fund-name URL is
replaced by this symbol-specific adapter. Current fields are never substituted
into historical CAPM or cash-rate calculations. The ^IRX quotation is not silently
treated as an effective annual cash return.

In [ ]:
network_provenance = {"enabled": bool(RUN_NETWORK_CONTEXT), "status": "not requested"}
if RUN_NETWORK_CONTEXT:
    import hashlib
    from datetime import datetime, timezone
    import yfinance as yf
    growth_indices = {"S&P 500": "^GSPC", "Nikkei 225": "^N225", "Hang Seng": "^HSI",
                      "Shanghai Composite": "000001.SS", "Euro Stoxx 50": "^STOXX50E", "Nifty 50": "^NSEI"}
    symbols = list(growth_indices.values()) + ["GLD", "^TNX", "^VIX"]
    context_raw = yf.download(symbols, start=NETWORK_CONTEXT_START, end=END_DATE,
                              auto_adjust=False, progress=False, threads=False, timeout=30,
                              multi_level_index=True)
    if context_raw.empty or "Close" not in context_raw:
        raise ValueError("Optional context download returned no closes.")
    context_prices = context_raw["Close"].reindex(columns=symbols)
    if (context_prices.notna().sum() < 2).any():
        raise ValueError("Optional context history is missing one or more requested series.")
    common_equities = context_prices[list(growth_indices.values())].dropna()
    if len(common_equities) < 24:
        raise ValueError("Insufficient common equity-index observations.")
    standardized_indices = common_equities.apply(normalize_context).rename(columns={v: k for k, v in growth_indices.items()})
    context_series = pd.concat([
        standardized_indices.mean(axis=1).rename("Standardized equity-index composite"),
        normalize_context(context_prices["GLD"]).rename("Gold ETF price: standardized"),
        normalize_context(context_prices["^TNX"]).rename("10-year Treasury yield: standardized"),
        normalize_context(context_prices["^VIX"]).rename("VIX level: standardized")], axis=1)
    diagnostic_tables.update(optional_market_closes=context_prices, optional_market_context=context_series,
                             optional_standardized_equity_indices=standardized_indices)
    show_panels(context_series, "Market-price context; full-sample normalization", "optional_market_context", "Standard deviations")
    metadata_rows = []
    for symbol in universe_prices:
        try:
            info = yf.Ticker(symbol).get_info()
            metadata_rows.append({"ETF": symbol, "Provider beta3Year": info.get("beta3Year", np.nan),
                                  "Provider totalAssets": info.get("totalAssets", np.nan), "Error": ""})
        except Exception as exc:
            metadata_rows.append({"ETF": symbol, "Provider beta3Year": np.nan,
                                  "Provider totalAssets": np.nan, "Error": type(exc).__name__})
    provider_metadata = pd.DataFrame(metadata_rows).set_index("ETF")
    provider_metadata["Initial-window SPY beta (different period)"] = capm_spy["Beta"]
    irx = yf.Ticker("^IRX").history(period="5d")
    if irx.empty or "Close" not in irx:
        raise ValueError("Recent ^IRX quote unavailable.")
    diagnostic_tables.update(optional_current_metadata=provider_metadata,
                             optional_recent_irx_quote=irx[["Close"]].rename(columns={"Close": "Annual discount yield quote (%)"}))
    network_provenance = {"enabled": True, "status": "downloaded", "downloaded_utc": datetime.now(timezone.utc).isoformat(),
                          "start": NETWORK_CONTEXT_START, "end_exclusive": END_DATE,
                          "context_prices_csv_sha256": hashlib.sha256(context_prices.to_csv().encode()).hexdigest(),
                          "metadata_period": "current provider snapshot; not historical", "auto_adjust": False}
    display(provider_metadata)
    display(diagnostic_tables["optional_recent_irx_quote"].tail(1))
else:
    print("Optional network context not requested. Enable RUN_NETWORK_CONTEXT to execute this section.")

### F. Source-code reconciliation

All **127 code cells** in the supplied **213-cell notebook** have been reviewed.
Indices below are zero-based positions in that supplied file, including Markdown
cells. Repeated displays, variable reshaping and repeated versions of a formula
are consolidated into one maintained calculation. Known incorrect calculations
are replaced explicitly; their original bytes remain in the historical materials.
No original research topic is silently omitted. Network-only inputs are executable
in the optional appendix and do not run by default.

The notebook metadata records a SHA-256 for the supplied file and each code cell,
plus its destination cell IDs. This mapping does not claim verbatim reproduction.

| Source cells | Topic | Destination cell IDs | Review decision |
| --- | --- | --- | --- |
| 5, 6, 7 | Setup and helpers | `imports`, `configuration` | Modern imports, numeric weights and labelled tables replace formatting-only helpers. |
| 10, 12, 14, 15 | Acquisition, cache and coverage | `prices`, `universe-diagnostics`, `network-context` | Frozen prices plus explicit Yahoo mode; optional metadata replaces the hard-coded scraper. |
| 19, 21, 22, 24, 25, 26, 27, 28, 38, 43, 44, 45, 81, 82, 83 | Return preparation and inspection | `universe-diagnostics`, `analysis` | Within-asset returns, gap rejection and outlier/coverage tables; repeated previews consolidated. |
| 30, 32, 34, 76, 80, 84, 85, 86, 87 | Beta, CAPM and expected-return comparison | `rolling-diagnostics`, `bl-details`, `network-context` | Matching market proxies and prior-window estimation; current metadata separate. |
| 37, 39, 40, 67, 69, 70 | Moments, covariance and correlation | `universe-diagnostics`, `bl-details` | Correct annual units and genuine correlation; selection remains declared retrospective. |
| 48, 49, 50, 51 | Rolling alpha/beta and window score | `rolling-diagnostics` | Compare conditional return residuals on common training dates, never betas versus returns. |
| 54, 55, 56, 57 | Performance, Sortino, growth and drawdown | `universe-diagnostics`, `bab-rolling` | Consistent period risk-free rate, downside denominator and initial capital. |
| 60, 63, 64, 73, 74 | AUM and capped weights | `aum-snapshot` | Snapshot-only appendix with repeated cap enforcement; no historical AUM claim. |
| 78, 116, 117, 119, 121, 123, 127, 128, 129, 130, 131, 132, 137, 139 | BL prior, P/Q/Omega and covariance | `bl-details` | View-scaled uncertainty and linear solves; redundant formula checks consolidated. |
| 89, 90, 91, 93, 95, 96, 134, 135, 142, 175 | Return-vector and risk-aversion allocations | `risk-aversion` | Covariance-aware optimization with cash replaces normalized returns and arbitrary weight division. |
| 100, 114 | Equity-index, gold, yield and VIX context | `network-context`, `macro-context` | Optional acquisition with honest proxy labels; no inputs fabricated. |
| 103, 104, 105, 106, 108, 109, 110, 112, 113 | Inflation and BIS credit | `macro-context` | Named/economic-key series selection; retain original units and standardized display. |
| 146, 147, 148, 150, 151, 153, 154, 155, 163, 164, 165, 166, 168, 169, 170, 171, 172 | Core objectives and comparison plots | `analysis`, `weights`, `metrics`, `charts`, `bab-rolling` | Actual variance, Sharpe and tracking objectives with consistent annual metrics; no relabelled variance optimizer. |
| 157, 158, 160 | Higher-order moments and utility | `higher-moments`, `analysis` | Raw central tensors checked against the optimizer own portfolio; standardized pair slices are descriptive. |
| 178, 179, 180, 181, 182, 183, 184, 185 | Five-factor data, regression and plots | `factor-analysis`, `factor-fit` | Monthly compounding and decimal factors; full five-factor fitted values; repeated previews consolidated. |
| 187, 189, 191, 193, 194, 195 | Factor-threshold heuristic | `factor-tilt-experiment` | Appendix sensitivity with capped weights and correctly annualized monthly calibration metrics. |
| 198, 199, 201, 202 | Quartile beta spread and rolling exposures | `beta-spread`, `bab-rolling` | Separate uncosted factor proxy and net beta-neutral strategy diagnostics. |
| 206, 207, 208, 209 | Kelly weights, fractions and wealth | `kelly-comparison` | Cash-aware constrained full Kelly then scaling; dated net simple-return ledger. |

## Save the complete research outputs

Each run creates a new timestamped folder under `outputs/`. Besides the main
portfolio results, it saves all restored diagnostic tables and figures, the raw
third/fourth-order moment tensors with asset ordering, source-cell reconciliation,
and optional acquisition status. These supplemental results are labelled as
diagnostics or sensitivity experiments in their notebook sections.

In [ ]:
output_folder = save_run(result, provenance, ANNUAL_RISK_FREE_RATE, MAX_WEIGHT)
if factor_results is not None:
    factor_results.to_csv(output_folder / "factor_regression.csv")
if universe_factor_results is not None:
    universe_factor_results.to_csv(output_folder / "universe_factor_regression.csv")
for label, table in diagnostic_tables.items():
    table.to_csv(output_folder / f"{label}.csv")
for label, figure in diagnostic_figures.items():
    figure.savefig(output_folder / f"{label}.png", dpi=120, bbox_inches="tight")
np.savez_compressed(output_folder / "higher_moment_tensors.npz",
                    third=moment_diagnostics["third"], fourth=moment_diagnostics["fourth"], assets=np.array(ASSETS))
weight_chart.savefig(output_folder / "allocations.png", dpi=150, bbox_inches="tight")
growth_chart.savefig(output_folder / "growth.png", dpi=150, bbox_inches="tight")
import json
import nbformat
notebook_source = nbformat.read("portfolio_construction.ipynb", as_version=4)
supplementary_info = {
    "rolling_window_candidates": list(ROLLING_WINDOWS), "selected_window": best_window,
    "risk_aversions": list(RISK_AVERSIONS), "kelly_comparison_fractions": list(KELLY_COMPARISON_FRACTIONS),
    "network_context": network_provenance, "source_reconciliation": dict(notebook_source.metadata.source_reconciliation),
    "diagnostic_tables": list(diagnostic_tables), "diagnostic_figures": list(diagnostic_figures),
    "interpretation": "Core diagnostics and explicitly labelled supplementary experiments; see notebook sections.",
}
(output_folder / "supplementary_info.json").write_text(json.dumps(supplementary_info, indent=2), encoding="utf-8")
print("RUN COMPLETE")
print("Data:", provenance["label"])
print("Saved results:", output_folder.resolve())